In [2]:
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
import tensorflow as tf
import numpy as np
import base64
from io import BytesIO
from PIL import Image
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import requests
import urllib3
from playwright.async_api import async_playwright
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Tắt cảnh báo khi truy cập các trang web có chứng chỉ SSL không hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ==========================================
# 0. CẤU HÌNH BẢO MẬT EMAIL (SMTP)
# ==========================================
# Nếu dùng Gmail (đã có mật khẩu ứng dụng):
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
SENDER_EMAIL = "nhatphivo781@gmail.com" 
SENDER_PASSWORD = "oavu cbps vvga npcg"

# Nếu dùng Mailtrap/Brevo (để test/tránh bị Google chặn), thay bằng thông số họ cấp:
# SMTP_SERVER = "sandbox.smtp.mailtrap.io"
# SMTP_PORT = 2525
# SENDER_EMAIL = "your_mailtrap_user"
# SENDER_PASSWORD = "your_mailtrap_password"

# ==========================================
# 1. KHỞI TẠO FASTAPI & SERVER
# ==========================================
nest_asyncio.apply() # Bắt buộc để chạy uvicorn trong Jupyter
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], 
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==========================================
# 2. TẢI MÔ HÌNH AI
# ==========================================
print("Đang tải model TensorFlow. Vui lòng đợi...")
model = tf.keras.models.load_model('models/web_defacement_model.keras')
print("✅ Tải model thành công!")

class DetectionRequest(BaseModel):
    type: str
    data: str
    model_name: str
    email: str

# ==========================================
# 3. HÀM GỬI EMAIL CẢNH BÁO
# ==========================================
def send_email_alert(target_email, source, confidence):
    try:
        msg = MIMEMultipart()
        msg['From'] = SENDER_EMAIL
        msg['To'] = target_email
        msg['Subject'] = f"🚨 CẢNH BÁO BẢO MẬT: Phát hiện tấn công Deface!"

        body = f"""
        Hệ thống phát hiện dấu hiệu bị thay đổi giao diện (Web Defacement):
        
        - Nguồn phát hiện: {source}
        - Mức độ nguy hiểm (AI dự đoán): {confidence}%
        
        Vui lòng kiểm tra lại hệ thống hoặc liên hệ quản trị viên ngay lập tức!
        """
        msg.attach(MIMEText(body, 'plain', 'utf-8'))

        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.sendmail(SENDER_EMAIL, target_email, msg.as_string())
        server.quit()
        print(f"📧 Đã gửi email cảnh báo tự động tới: {target_email}")
    except Exception as e:
        print(f"❌ Lỗi khi gửi email: {e}")

# ==========================================
# 4. HÀM TIỀN XỬ LÝ ẢNH
# ==========================================
def process_image(img):
    img_array = np.array(img)
    img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)
    img_tensor = tf.keras.preprocessing.image.smart_resize(img_tensor, (224, 224))
    img_tensor = tf.expand_dims(img_tensor, axis=0)
    return img_tensor

def process_base64_image(base64_str):
    image_data = base64.b64decode(base64_str.split(',')[1])
    img = Image.open(BytesIO(image_data)).convert('RGB')
    return process_image(img)

def process_bytes_image(image_bytes):
    img = Image.open(BytesIO(image_bytes)).convert('RGB')
    return process_image(img)

# ==========================================
# 5. HÀM CHỤP MÀN HÌNH TỪ URL THÔNG MINH
# ==========================================
async def capture_screenshot_from_url(url: str):
    if not url.startswith("http://") and not url.startswith("https://"):
        url = "http://" + url
        
    try:
        response = requests.get(url, verify=False, timeout=10)
        content_type = response.headers.get('Content-Type', '').lower()
        if 'image' in content_type:
            print(f"📸 URL là file ảnh trực tiếp ({content_type}). Đang tải...")
            return response.content
    except Exception:
        pass # Bỏ qua nếu lỗi, chuyển sang dùng Playwright

    print("🌐 Truy cập URL bằng trình duyệt ảo Playwright...")
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            context = await browser.new_context(
                ignore_https_errors=True,
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
            )
            page = await context.new_page()
            await page.set_viewport_size({"width": 1366, "height": 768})
            
            try:
                await page.goto(url, wait_until="load", timeout=30000)
            except Exception:
                pass # Ép chụp kể cả khi trang tải lỗi một phần
            
            await page.wait_for_timeout(2000)
            screenshot_bytes = await page.screenshot()
            await browser.close()
            return screenshot_bytes
            
    except Exception as e:
        print(f"❌ Lỗi Playwright: {e}")
        return None

# ==========================================
# 6. API ENDPOINT CHÍNH
# ==========================================
@app.post("/detect")
async def detect_deface(request: DetectionRequest):
    try:
        source_name = "Ảnh chụp trực tiếp từ Extension"
        
        # Phân loại request
        if request.type == 'image':
            img_tensor = process_base64_image(request.data)
            
        elif request.type == 'url':
            source_name = request.data
            print(f"\n--- NHẬN YÊU CẦU QUÉT URL: {source_name} ---")
            screenshot_bytes = await capture_screenshot_from_url(source_name)
            
            if not screenshot_bytes:
                return {"error": "Trang web không tồn tại, máy chủ bị sập, hoặc kết nối quá hạn!"}
            img_tensor = process_bytes_image(screenshot_bytes)
            
        else:
            return {"error": "Loại request không hợp lệ!"}

        # Đưa vào Model để dự đoán
        prediction = model.predict(img_tensor, verbose=0)
        deface_probability = float(prediction[0][1]) 
        
        # Áp dụng ngưỡng cảnh báo
        NGUONG_CANH_BAO = 0.65 
        is_defaced = bool(deface_probability > NGUONG_CANH_BAO)
        
        # Tính toán độ tự tin
        confidence = round(deface_probability * 100 if is_defaced else (1 - deface_probability) * 100, 2)
        
        # Gửi email nếu có deface và người dùng có nhập email
        if is_defaced and request.email:
            send_email_alert(request.email, source_name, confidence)
        
        return {
            "is_defaced": is_defaced,
            "confidence": confidence,
            "message": "Phân tích hoàn tất"
        }
            
    except Exception as e:
        return {"error": f"Lỗi phía server: {str(e)}"}

# ==========================================
# 7. LỆNH KHỞI CHẠY SERVER
# ==========================================
if __name__ == "__main__":
    print("\n" + "="*50)
    print("🚀 ĐANG KHỞI ĐỘNG BACKEND SERVER TẠI HTTP://127.0.0.1:8000")
    print("⏹️ Nhấn nút 'Interrupt' (hoặc bấm 0, 0) nếu muốn tắt server.")
    print("="*50 + "\n")
    
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    await server.serve()

Đang tải model TensorFlow. Vui lòng đợi...
✅ Tải model thành công!

🚀 ĐANG KHỞI ĐỘNG BACKEND SERVER TẠI HTTP://127.0.0.1:8000
⏹️ Nhấn nút 'Interrupt' (hoặc bấm 0, 0) nếu muốn tắt server.

📧 Đã gửi email cảnh báo tự động tới: trangiaphu39@gmail.com
